# Biblical Cross-Encoder Fine-Tuning (Save to Drive)

This notebook fine-tunes a Cross-Encoder model specifically for "Pastor Paraphrase" matching and saves all outputs directly to your Google Drive.

**Goal:** Transform colloquial phrases like *"Peter sinking in the waves"* into a precision match for *Matthew 14:30*.

**Runtime Requirement:** GPU (T4 or better).

In [ ]:
# 1. Setup Google Drive & Install dependencies
from google.colab import drive
import os
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/BiblePresenter_Training'
os.makedirs(OUTPUT_DIR, exist_ok=True)

!pip install -q sentence-transformers optimum[onnxruntime] torch pandas sqlite3

In [ ]:
# 2. Download Bible Data
import urllib.request
import sqlite3
import pandas as pd

DB_URL = 'https://raw.githubusercontent.com/alshival/super_bible/main/SUPER_BIBLE/super_bible.db'
urllib.request.urlretrieve(DB_URL, 'super_bible.db')

conn = sqlite3.connect('super_bible.db')
df = pd.read_sql_query("SELECT title, chapter, verse, text FROM super_bible WHERE language='EN' AND version='KJV'", conn)
conn.close()

print(f"Loaded {len(df)} verses for training.")

In [ ]:
# 3. Synthetic Data Generation
from sentence_transformers import InputExample
import json

train_samples = []
dataset_for_saving = []

def create_paraphrases(verse_text):
    clean = verse_text.replace("thee", "you").replace("thou", "you").replace("shall", "will").replace("unto", "to")
    return [
        f"that part where it says {clean[:50]}...",
        f"{clean.lower()}",
        f"the verse about {clean.split()[-1]} and {clean.split()[0]}"
    ]

target_verses = df.sample(2000).to_dict('records')

for row in target_verses:
    verse_text = row['text']
    ref = f"{row['title']} {row['chapter']}:{row['verse']}"
    paras = create_paraphrases(verse_text)
    
    for p in paras:
        train_samples.append(InputExample(texts=[p, verse_text], label=1.0))
        dataset_for_saving.append({"query": p, "passage": verse_text, "label": 1.0})
        
    wrong_verse = df[df['title'] == row['title']].sample(1).iloc[0]['text']
    if wrong_verse != verse_text:
        train_samples.append(InputExample(texts=[paras[0], wrong_verse], label=0.0))
        dataset_for_saving.append({"query": paras[0], "passage": wrong_verse, "label": 0.0})

dataset_path = os.path.join(OUTPUT_DIR, 'training_dataset.json')
with open(dataset_path, 'w') as f:
    json.dump(dataset_for_saving, f, indent=2)

print(f"Generated {len(train_samples)} training pairs.")
print(f"Dataset saved to Drive: {dataset_path}")

In [ ]:
# 4. Training
from sentence_transformers import CrossEncoder
from torch.utils.data import DataLoader

model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', num_labels=1, device='cuda')
train_dataloader = DataLoader(train_samples, shuffle=True, batch_size=16)

model_save_path = os.path.join(OUTPUT_DIR, 'biblical_reranker_pt')

print("Starting Fine-tuning...")
model.fit(train_dataloader=train_dataloader,
          epochs=1,
          warmup_steps=100,
          output_path=model_save_path)
print(f"Training Complete. PyTorch model saved to Drive: {model_save_path}")

In [ ]:
# 5. Export to ONNX directly to Drive
print("Exporting to ONNX...")
onnx_output_path = os.path.join(OUTPUT_DIR, 'model_onnx')
!optimum-cli export onnx --model {model_save_path} --task text-classification {onnx_output_path}
print(f"Export complete. ONNX files are in Drive: {onnx_output_path}")